# 02. Curules del Concejo de Barrancabermeja


Este cuaderno asigna las curules del Concejo mediante la cifra repartidora, compara el cálculo con los resultados publicados y describe la competencia electoral.

## Entradas y salidas

- Lee las tablas de `datos/procesados/votos/`.
- Escribe los resultados en `datos/procesados/concejo/` y la auditoría en `docs/fuentes/auditorias/02_curules/`.
- No modifica las salidas del cuaderno 01.

## Asignación de curules

Artículo 263 de la Constitución, modificado por el Acto Legislativo 01 de 2009:

1. **Votos válidos** = votos por listas más votos en blanco. Los votos nulos y no marcados se excluyen.
2. **Cuociente electoral** = votos válidos ÷ curules.
3. **Umbral** = 50 % del cuociente. Las listas que no lo alcanzan quedan fuera del reparto.
4. **Cifra repartidora:** se divide la votación de cada lista habilitada entre 1, 2, 3 y los divisores siguientes hasta el número de curules. El cociente que ocupa la posición de la última curul es la cifra repartidora.
5. **Curules de cada lista** = parte entera de (votos de la lista ÷ cifra repartidora).
6. **Asignación dentro de la lista:** en listas con voto preferente, las curules corresponden a las candidaturas con más votos. En listas cerradas, se sigue el orden de inscripción.

La curul se asigna primero a la lista y luego a sus candidaturas. Por eso, una candidatura con más votos individuales puede quedar por fuera si su lista no obtiene curul.

En una lista cerrada, los resultados no identifican quién ocupa la curul. Se requiere el formulario de inscripción. Mientras no esté disponible, la curul se asigna a la lista y el nombre queda vacío.

In [1]:
# --- 1. Configuración ----------------------------------------------------
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


def raiz_proyecto(inicio: Path) -> Path:
    for c in (inicio.resolve(), *inicio.resolve().parents):
        if (c / "datos").is_dir() and (c / "notebooks").is_dir():
            return c
    raise FileNotFoundError("Abre el cuaderno dentro de Proy_Electoral_Bca.")


RAIZ = raiz_proyecto(Path.cwd())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.analisis import asignacion as asg  # noqa: E402

PROCESADOS = RAIZ / "datos" / "procesados"
VOTOS = PROCESADOS / "votos"
SALIDA = PROCESADOS / "concejo"
AUDITORIAS = RAIZ / "docs" / "fuentes" / "auditorias" / "02_curules"

ID_EJECUCION = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
AUDITORIA = AUDITORIAS / ID_EJECUCION
AUDITORIA.mkdir(parents=True, exist_ok=True)
SALIDA.mkdir(parents=True, exist_ok=True)

VERSION_NB = "02.1.0"

# El número de concejales lo fija la ley según la población del municipio: la Ley 136 de
# 1994 asigna 17 a los municipios de más de 100.000 y hasta 250.000 habitantes.
# Barrancabermeja está en ese rango, y el Concejo actual en efecto tiene 17 miembros.
CURULES_CONCEJO = 17

OPCIONES_CSV = {"index": False, "float_format": "%.6f", "lineterminator": "\n",
                "encoding": "utf-8-sig"}

try:
    display
except NameError:  # el cuaderno también debe poder correrse como script
    def display(x):  # noqa: D103
        print(x.to_string(index=False) if hasattr(x, "to_string") else x)


def rel(p: Path) -> str:
    try:
        return Path(p).resolve().relative_to(RAIZ).as_posix()
    except ValueError:
        return Path(p).as_posix()


def escribir_csv(df: pd.DataFrame, ruta: Path) -> Path:
    ruta.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(ruta, **OPCIONES_CSV)
    return ruta


def escribir_json(ruta: Path, obj) -> Path:
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_text(json.dumps(obj, ensure_ascii=False, indent=2, sort_keys=True,
                               default=str), encoding="utf-8")
    return ruta


def miles(n) -> str:
    return f"{int(n):,}".replace(",", ".")


print("Proyecto: ", RAIZ)
print("Ejecución:", ID_EJECUCION)
print("Curules del Concejo:", CURULES_CONCEJO)

Proyecto:  D:\Users\LAURA PEREZ\Desktop\CIENCIA DE DATOS\Proy_Electoral_Bca
Ejecución: 20261003_235924_506795
Curules del Concejo: 17


## 2 · Las tablas de Concejo que dejó el cuaderno 01

In [2]:
def cargar(ruta: Path) -> pd.DataFrame:
    d = pd.read_parquet(ruta) if ruta.suffix == ".parquet" else pd.read_csv(ruta, dtype=str)
    d["votos"] = pd.to_numeric(d["votos"], errors="coerce").fillna(0).astype(int)
    return d


archivos = sorted(p for p in VOTOS.rglob("*concejo*") if p.suffix in (".csv", ".parquet"))
if not archivos:
    raise FileNotFoundError(
        f"No hay tablas de Concejo en {rel(VOTOS)}. Ejecuta primero el cuaderno 01."
    )

concejo = {}
for p in archivos:
    d = cargar(p)
    anio = str(d["anio"].iloc[0]) if "anio" in d.columns else p.parent.name
    concejo[anio] = d

inventario = pd.DataFrame([
    {"anio": a, "filas": len(d), "votos": int(d["votos"].sum()),
     "mesas": int(d[["cod_zona", "cod_puesto", "puesto", "mesa"]].drop_duplicates().shape[0]),
     "listas": int(d["cod_partido"].nunique())}
    for a, d in sorted(concejo.items())
])
display(inventario)
print(f"\nAños disponibles: {', '.join(sorted(concejo))}")

,anio,filas,votos,mesas,listas
0,2019,51457,121765,525,19
1,2023,61400,131961,593,26



Años disponibles: 2019, 2023


## 3 · El reparto de curules, paso a paso

Se muestra cada número intermedio a propósito. Un resultado electoral que no se puede
recorrer paso a paso no se puede defender.

In [3]:
resultados = {}
for anio in sorted(concejo):
    r = asg.asignar_curules(concejo[anio], CURULES_CONCEJO)
    resultados[anio] = r
    print(f"\n{'=' * 66}\nCONCEJO {anio}\n{'=' * 66}")
    print(f"  votos a listas        {miles(r['votos_a_listas']):>10}")
    print(f"  votos en blanco       {miles(r['votos_en_blanco']):>10}  (sí son válidos)")
    print(f"  VOTOS VÁLIDOS         {miles(r['votos_validos']):>10}")
    print(f"  votos nulos           {miles(r['votos_nulos']):>10}  (no son válidos)")
    print(f"  votos no marcados     {miles(r['votos_no_marcados']):>10}  (no son válidos)")
    print(f"\n  cuociente electoral   {r['cuociente_electoral']:>10,.1f}".replace(",", "."))
    print(f"  umbral (50%)          {r['umbral']:>10,.1f}".replace(",", "."))
    print(f"  CIFRA REPARTIDORA     {r['cifra_repartidora']:>10,.2f}".replace(",", "."))
    print(f"\n  listas con votos      {len(r['listas']):>10}")
    print(f"  pasan el umbral       {int(r['listas']['supera_umbral'].sum()):>10}")
    print(f"  obtienen curul        {int((r['listas']['curules'] > 0).sum()):>10}")
    print(f"  curules asignadas     {r['curules_asignadas']:>10}")


CONCEJO 2019
  votos a listas           111.489
  votos en blanco            4.428  (sí son válidos)
  VOTOS VÁLIDOS            115.917
  votos nulos                3.342  (no son válidos)
  votos no marcados          2.506  (no son válidos)

  cuociente electoral      6.818.6
  umbral (50%)             3.409.3
  CIFRA REPARTIDORA       4.412.00

  listas con votos              18
  pasan el umbral               13
  obtienen curul                11
  curules asignadas             17

CONCEJO 2023
  votos a listas           119.854
  votos en blanco            5.207  (sí son válidos)
  VOTOS VÁLIDOS            125.061
  votos nulos                3.246  (no son válidos)
  votos no marcados          3.654  (no son válidos)

  cuociente electoral      7.356.5
  umbral (50%)             3.678.3
  CIFRA REPARTIDORA       4.042.00

  listas con votos              25
  pasan el umbral               12
  obtienen curul                11
  curules asignadas             17


In [4]:
for anio, r in sorted(resultados.items()):
    t = r["listas"][r["listas"]["curules"] > 0].copy()
    t["votos_por_curul"] = (t["votos"] / t["curules"]).round(0).astype(int)
    print(f"\nListas con curul · Concejo {anio}")
    display(t[["partido", "votos", "curules", "votos_por_curul", "lista_cerrada"]]
            .sort_values("curules", ascending=False))
    escribir_csv(r["listas"], SALIDA / f"listas_concejo_{anio}.csv")
    escribir_csv(r["electos"], SALIDA / f"electos_concejo_{anio}.csv")


Listas con curul · Concejo 2019


,partido,votos,curules,votos_por_curul,lista_cerrada
0,PARTIDO CAMBIO RADICAL,13236,3,4412,False
1,PARTIDO LIBERAL COLOMBIANO,12574,2,6287,False
2,MOVIMIENTO ALTERNATIVO INDIGENA Y SOCIAL MAIS,11720,2,5860,False
3,SUMEMOS,9665,2,4832,False
4,PARTIDO ALIANZA SOCIAL INDEPENDIENTE ASI,9243,2,4622,False
5,CIVICOS AL 100%,8641,1,8641,False
6,PARTIDO ALIANZA VERDE,8208,1,8208,False
7,PARTIDO POLO DEMOCRATICO ALTERNATIVO,8020,1,8020,False
8,PARTIDO SOCIAL DE UNIDAD NACIONAL PARTIDO DE ...,5085,1,5085,False
9,BIENESTAR Y UNIDAD SOCIAL - BUS,4464,1,4464,False



Listas con curul · Concejo 2023


,partido,votos,curules,votos_por_curul,lista_cerrada
0,AGRUPACIÓN POLÍTICA EN MARCHA,15354,3,5118,False
1,CIVICOS,10842,2,5421,False
2,PARTIDO LIBERAL COLOMBIANO,10024,2,5012,False
3,INDEPENDIENTES,9295,2,4648,False
4,PARTIDO NUEVO LIBERALISMO,8084,2,4042,False
5,"MOVIMIENTO ALTERNATIVO INDÍGENA Y SOCIAL ""MAIS""",6757,1,6757,False
6,ENERGIA NUEVA,6648,1,6648,False
7,PARTIDO DEMÓCRATA COLOMBIANO,6267,1,6267,False
8,PARTIDO POLÍTICO LA FUERZA DE LA PAZ,5600,1,5600,False
9,PACTO HISTORICO,4111,1,4111,True


## 4 · Validación contra el Concejo real

Un cálculo electoral que no se contrasta contra el resultado oficial no vale nada. Aquí
se compara contra los nombres que publica el Concejo Distrital, y **las diferencias se
explican una por una** en lugar de quedar como ruido.

In [5]:
# Concejales que publica el Concejo Distrital de Barrancabermeja para el período
# 2024-2027. Fuente: concejobarrancabermeja.gov.co, consultado en octubre de 2026.
CONCEJO_PUBLICADO_2024_2027 = [
    "Norbeiro Certuche Vasquez", "Nestor Robert Alvarez Moreno",
    "Anastasia Martinez Becerra", "Jhon Blert Corena Ahumada",
    "Ariel Zambrano Gonzalez", "Ceferino Garay Caballero",
    "Maria Eugenia Niño Gutierrez", "Gabriel Bohorquez Gonzalez",
    "Ivan Jesus Serrano Miranda", "Jasser Cruz Gambindo",
    "Jorge Ivan Cortinez Alarcon", "Jorge Luis Escalante Viana",
    "Wbeimar Guiza Muñoz", "Josue Jovid Taborda Moreno",
    "Sandra Milena Galvis Mora", "Sebastian Rodriguez",
    "Tania Andrea Mogollon Zapata",
]

if "2023" in resultados:
    comp = asg.comparar_con_lista_oficial(resultados["2023"], CONCEJO_PUBLICADO_2024_2027)
    print(f"Curules calculadas:                {comp['curules_calculadas']}")
    print(f"  con nombre (voto preferente):    {comp['curules_con_nombre']}")
    print(f"  sin nombre (lista cerrada):      {comp['curules_sin_nombre_por_lista_cerrada']}")
    print(f"\nCoinciden con el Concejo publicado: {comp['coinciden']} de "
          f"{comp['nombres_oficiales']}")
    if comp["solo_en_el_calculo"]:
        print("\nSalen en el cálculo y no en la lista publicada:")
        for n in comp["solo_en_el_calculo"]:
            print(f"  · {n}")
    if comp["solo_en_la_lista_oficial"]:
        print("\nSalen en la lista publicada y no en el cálculo:")
        for n in comp["solo_en_la_lista_oficial"]:
            print(f"  · {n}")
    escribir_json(AUDITORIA / "validacion_contra_concejo_publicado.json", comp)

Curules calculadas:                17
  con nombre (voto preferente):    16
  sin nombre (lista cerrada):      1

Coinciden con el Concejo publicado: 14 de 17

Salen en el cálculo y no en la lista publicada:
  · ALEIDA NOGUERA PUERTO
  · JOSE ARMANDO JAIME OVIEDO

Salen en la lista publicada y no en el cálculo:
  · Norbeiro Certuche Vasquez
  · Jasser Cruz Gambindo
  · Wbeimar Guiza Muñoz


### 4.1 · Por qué difieren esas curules

Las diferencias no son errores del cálculo. Tienen tres causas distintas y conviene no
confundirlas, porque cada una dice algo diferente sobre los datos.

**Una curul sin nombre: el Pacto Histórico llevó lista cerrada.** Toda su votación fue a
la lista y ningún candidato suyo tiene votación propia, así que el archivo de resultados
no dice quién se posesiona. En los reportes oficiales su electo figura con **cero votos**,
precisamente por eso. Para resolverlo hace falta el formulario de candidaturas inscritas,
que está en la lista de fuentes pendientes. Mientras tanto, la curul queda asignada al
partido y el nombre en blanco.

**Dos nombres cambiados: la lista publicada es el Concejo de hoy, no el de la elección.**
Entre la declaratoria de 2023 y hoy hubo renuncias y reemplazos. El cálculo reproduce
quién **fue elegido**; la página del Concejo muestra quién **está sentado**. Son dos
preguntas distintas y las dos son correctas.

El cuadro siguiente deja la diferencia a la vista: en cada pareja, quien sale del cálculo
tiene más votos que quien figura hoy, y ambos son del mismo partido.

In [6]:
if "2023" in resultados:
    d23 = concejo["2023"]
    cand23 = (d23[d23["clase_opcion"] == "candidatura"]
              .groupby(["partido", "candidato"], as_index=False)["votos"].sum())
    interes = ["NOGUERA", "CERTUCHE VASQUEZ", "JAIME OVIEDO", "GUIZA MUÑOZ"]
    sel = cand23[cand23["candidato"].map(asg.canon).str.contains("|".join(interes))]
    display(sel.sort_values(["partido", "votos"], ascending=[True, False]))
    print("\nEn cada partido, quien el cálculo declara electo tiene más votos que quien\n"
          "figura hoy en el Concejo. Eso es un reemplazo posterior a la elección, no un\n"
          "error de la fórmula.")

,partido,candidato,votos
71,INDEPENDIENTES,JOSE ARMANDO JAIME OVIEDO,1149
308,PARTIDO POLÍTICO LA FUERZA DE LA PAZ,ALEIDA NOGUERA PUERTO,1480
321,PARTIDO POLÍTICO LA FUERZA DE LA PAZ,NORBEIRO CERTUCHE VASQUEZ,880



En cada partido, quien el cálculo declara electo tiene más votos que quien
figura hoy en el Concejo. Eso es un reemplazo posterior a la elección, no un
error de la fórmula.


## 5 · El efecto de la cifra repartidora

Esta es la sección que responde la pregunta que la gente hace en voz alta: **¿cómo es que
fulano sacó más votos y no quedó?**

La respuesta es que la curul no la gana el candidato: la gana la lista. El candidato
compite dos veces, primero contra el país entero por el umbral de su lista, y después
contra sus propios compañeros por el puesto dentro de ella.

In [7]:
for anio in sorted(resultados):
    u = asg.umbral_real_de_entrada(resultados[anio], concejo[anio])
    if u.get("sin_datos"):
        continue
    print(f"\n{'=' * 66}\nCONCEJO {anio}")
    print(f"  el primer electo entró con   {miles(u['votos_del_primer_electo']):>8} votos")
    print(f"  el último electo entró con   {miles(u['votos_del_ultimo_electo']):>8} votos")
    print(f"\n  candidatos con MÁS votos que el último electo que NO entraron: "
          f"{u['no_electos_con_mas_votos_que_el_ultimo_electo']}")
    t = u["tabla_mejores_perdedores"]
    if len(t):
        display(t[["partido", "candidato", "votos"]].head(10))
        escribir_csv(t, SALIDA / f"no_electos_con_mas_votos_{anio}.csv")


CONCEJO 2019
  el primer electo entró con      2.684 votos
  el último electo entró con      1.013 votos

  candidatos con MÁS votos que el último electo que NO entraron: 18


,partido,candidato,votos
2,PARTIDO LIBERAL COLOMBIANO,MORGAN EGEA SANCHEZ,1705
42,PARTIDO CAMBIO RADICAL,DEBISON GOMEZ MARTINEZ,1614
11,PARTIDO LIBERAL COLOMBIANO,ALFONSO BAEZA ACUÑA,1528
4,PARTIDO LIBERAL COLOMBIANO,FRANKLIN ANGARITA BECERRA,1512
35,PARTIDO CAMBIO RADICAL,HOBER TORRES ARRIETA,1489
54,PARTIDO ALIANZA VERDE,LUIS FERNANDO CALDERON MEJIA,1447
0,PARTIDO LIBERAL COLOMBIANO,YEISY MARIA GONZALEZ MARTINEZ,1407
170,MOVIMIENTO ALTERNATIVO INDIGENA Y SOCIAL MAIS,JHON BLERT CORENA AHUMADA,1332
190,CIVICOS AL 100%,CAMILA MARCELA GONZALEZ GALINDO,1273
51,PARTIDO ALIANZA VERDE,CESAR AUGUSTO GUZMAN AREIZA,1214



CONCEJO 2023
  el primer electo entró con      2.228 votos
  el último electo entró con        932 votos

  candidatos con MÁS votos que el último electo que NO entraron: 21


,partido,candidato,votos
310,VAMOS UNIDOS,KELLY ZULIMA ORTIZ CALAO,1472
229,LOS VERIDICOS,ELVER MARIN QUIROGA,1412
251,CIVICOS,STEVEN ANDRES ZAMBRANO MOLINA,1395
183,AGRUPACIÓN POLÍTICA EN MARCHA,JOSE ELIAS ZORRO MANRIQUE,1384
182,AGRUPACIÓN POLÍTICA EN MARCHA,LA CHIQUI CARMENZA SANTIAGO OSPINO,1291
185,AGRUPACIÓN POLÍTICA EN MARCHA,JOHN JAIRO GARCIA GONZALEZ,1263
339,JUNTOS,CESAR AUGUSTO GUZMAN AREIZA,1186
176,AGRUPACIÓN POLÍTICA EN MARCHA,CRISTIAN MAURICIO RAMIREZ ARIAS,1184
203,INDEPENDIENTES,WBEIMAR GUIZA MUÑOZ,1145
218,INDEPENDIENTES,MARCO GREGORIO LARA JAIMES,1140


## 6 · Los votos que no eligieron a nadie

Cuántos votos fueron a listas que no alcanzaron ninguna curul. No es una irregularidad:
es cómo funciona un sistema con umbral. Pero es una cifra que la ciudadanía casi nunca ve.

In [8]:
desperdicio = []
for anio in sorted(resultados):
    p = asg.votos_sin_representacion(resultados[anio])
    desperdicio.append({
        "anio": anio,
        "listas_sin_curul": p["listas_sin_curul"],
        "votos_sin_representacion": p["votos_en_listas_sin_curul"],
        "pct_de_los_validos": p["porcentaje_sobre_validos"],
    })
    print(f"\nConcejo {anio}: {miles(p['votos_en_listas_sin_curul'])} votos "
          f"({p['porcentaje_sobre_validos']} % de los válidos) fueron a "
          f"{p['listas_sin_curul']} listas que no obtuvieron ninguna curul.")
    display(p["tabla"].head(6)[["partido", "votos", "supera_umbral"]])

tabla_desperdicio = pd.DataFrame(desperdicio)
escribir_csv(tabla_desperdicio, SALIDA / "votos_sin_representacion.csv")


Concejo 2019: 16.179 votos (13.96 % de los válidos) fueron a 7 listas que no obtuvieron ninguna curul.


,partido,votos,supera_umbral
11,PARTIDO CENTRO DEMOCRATICO,4341,True
12,BARRANCABERMEJA LIBRE HUMANA Y EN PAZ,3600,True
13,PARTIDO CONSERVADOR COLOMBIANO,3037,False
14,PARTIDO ADA,2372,False
15,PARTIDO COLOMBIA JUSTA LIBRES,1122,False
16,MOVIMIENTO AUTORIDADES INDIGENAS DE COLOMBIA AICO,886,False



Concejo 2023: 32.799 votos (26.23 % de los válidos) fueron a 14 listas que no obtuvieron ninguna curul.


,partido,votos,supera_umbral
11,PARTIDO CONSERVADOR - PARTIDO MIRA,3853,True
12,"MOVIMIENTO AUTORIDADES INDÍGENAS DE COLOMBIA ""...",3644,False
13,PARTIDO COLOMBIA RENACIENTE,3412,False
14,FUERZA CON DIGNIDAD,3134,False
15,JUNTOS,3096,False
16,LOS VERIDICOS,3095,False


WindowsPath('D:/Users/LAURA PEREZ/Desktop/CIENCIA DE DATOS/Proy_Electoral_Bca/datos/procesados/concejo/votos_sin_representacion.csv')

## 7 · La forma de la competencia

Tres medidas estándar, las mismas que se usan en la literatura comparada para que el
resultado sea contrastable con otros municipios y otros países.

- **Número efectivo de partidos** (Laakso y Taagepera): cuántos partidos pesan de verdad,
  que casi nunca es cuántos se presentaron.
- **HHI**: concentración, de 0 a 1. Más alto, más concentrado.
- **Desproporcionalidad de Gallagher**: cuánto se aparta el reparto de curules del reparto
  de votos. Es la medida usual para evaluar un sistema electoral.

In [9]:
indicadores = []
for anio in sorted(resultados):
    i = asg.indicadores_competencia(resultados[anio])
    i["anio"] = anio
    i["cifra_repartidora"] = round(resultados[anio]["cifra_repartidora"], 2)
    i["umbral"] = round(resultados[anio]["umbral"], 1)
    indicadores.append(i)

tabla_ind = pd.DataFrame(indicadores).set_index("anio").T
display(tabla_ind)
escribir_csv(pd.DataFrame(indicadores), SALIDA / "indicadores_competencia.csv")

anio,2019,2023
listas_inscritas_con_votos,18.0000,25.0000
listas_que_superaron_el_umbral,13.0000,12.0000
listas_con_al_menos_una_curul,11.0000,11.0000
numero_efectivo_de_partidos_por_votos,12.8000,16.0000
numero_efectivo_de_partidos_por_curules,9.3200,9.3200
hhi_votos,0.0781,0.0625
hhi_curules,0.1073,0.1073
desproporcionalidad_gallagher,7.5100,9.1100
cifra_repartidora,4412.0000,4042.0000
umbral,3409.3000,3678.3000


WindowsPath('D:/Users/LAURA PEREZ/Desktop/CIENCIA DE DATOS/Proy_Electoral_Bca/datos/procesados/concejo/indicadores_competencia.csv')

## 8 · Dónde se vota a quién

El voto no se reparte igual por la ciudad. Esta sección mide, para cada electo, qué tan
concentrada estuvo su votación en el territorio.

Se agrupa por `id_puesto`, **nunca por `cod_puesto`**: el cuaderno 01 mostró que el código
de puesto no identifica el puesto en todos los años.

In [10]:
territorio = []
for anio in sorted(resultados):
    d = concejo[anio]
    electos = resultados[anio]["electos"]
    con_nombre = electos[electos["clase"] == "voto_preferente"]
    if con_nombre.empty:
        continue
    votos_electos = d[d["cod_candidato"].isin(con_nombre["cod_candidato"])
                      & d["cod_partido"].isin(con_nombre["cod_partido"])]
    for r in con_nombre.to_dict("records"):
        sub = d[(d["cod_partido"] == r["cod_partido"])
                & (d["cod_candidato"] == r["cod_candidato"])]
        if sub.empty or sub["votos"].sum() == 0:
            continue
        por_puesto = sub.groupby("id_puesto")["votos"].sum().sort_values(ascending=False)
        total = int(por_puesto.sum())
        # Concentración: qué parte de su votación salió de sus 3 mejores puestos.
        territorio.append({
            "anio": anio, "partido": r["partido"], "candidato": r["candidato"],
            "votos": total,
            "puestos_con_voto": int((por_puesto > 0).sum()),
            "pct_en_sus_3_mejores_puestos": round(por_puesto.head(3).sum() / total * 100, 1),
            "hhi_territorial": round(float(((por_puesto / total) ** 2).sum()), 4),
        })

if territorio:
    t = pd.DataFrame(territorio).sort_values(
        ["anio", "pct_en_sus_3_mejores_puestos"], ascending=[True, False])
    display(t)
    escribir_csv(t, SALIDA / "concentracion_territorial_electos.csv")
    print("\nCómo se lee: un valor alto en 'pct_en_sus_3_mejores_puestos' indica una "
          "votación\nconcentrada en pocos lugares; uno bajo, una votación repartida por "
          "toda la ciudad.\nLas dos cosas son formas legítimas de hacer campaña y ninguna "
          "es, por sí sola,\nindicio de nada.")

,anio,partido,candidato,votos,puestos_con_voto,pct_en_sus_3_mejores_puestos,hhi_territorial
16,2019,BIENESTAR Y UNIDAD SOCIAL - BUS,LUIS ENRIQUE SANCHEZ PALOMINO,1013,43,75.1,0.3548
0,2019,PARTIDO POLO DEMOCRATICO ALTERNATIVO,JASER CRUZ GAMBINDO,2684,60,32.0,0.0563
11,2019,PARTIDO ALIANZA SOCIAL INDEPENDIENTE ASI,JULIETT MARCELA RODRIGUEZ RINCON,1552,58,27.0,0.0488
12,2019,MOVIMIENTO ALTERNATIVO INDIGENA Y SOCIAL MAIS,LUIS MANUEL TORO HERNANDEZ,1365,56,26.7,0.0510
15,2019,MOVIMIENTO SOCIAL TAREA DE TODOS,JOHN JAIRO GARCIA GONZALEZ,1197,55,22.6,0.0366
1,2019,PARTIDO CAMBIO RADICAL,DARINEL VILLAMIZAR RUIZ,2383,61,17.1,0.0290
8,2019,SUMEMOS,LUIS ALBERTO ARISMENDI SOLANO,1701,60,16.6,0.0286
4,2019,PARTIDO LIBERAL COLOMBIANO,EDGARDO MOSCOTE PABA,2037,61,16.5,0.0290
13,2019,MOVIMIENTO ALTERNATIVO INDIGENA Y SOCIAL MAIS,EDSON LEONIDAS RUEDA RUEDA,1363,60,16.3,0.0300
10,2019,SUMEMOS,ERLIG DIANA JIMENEZ BECERRA,1596,63,16.1,0.0292



Cómo se lee: un valor alto en 'pct_en_sus_3_mejores_puestos' indica una votación
concentrada en pocos lugares; uno bajo, una votación repartida por toda la ciudad.
Las dos cosas son formas legítimas de hacer campaña y ninguna es, por sí sola,
indicio de nada.


### 8.1 · El voto por comuna

Reparto de la votación total del Concejo por comuna, con la advertencia que exige la
metodología del proyecto.

In [11]:
for anio in sorted(resultados):
    d = concejo[anio]
    col = "comuna_normalizada" if "comuna_normalizada" in d.columns else "comuna"
    c = (d.groupby([ "cod_comuna", col], as_index=False)["votos"].sum()
         .sort_values("votos", ascending=False))
    c["pct"] = (c["votos"] / c["votos"].sum() * 100).round(1)
    print(f"\nConcejo {anio} · votación por comuna")
    display(c.head(20))
    escribir_csv(c, SALIDA / f"votos_por_comuna_concejo_{anio}.csv")

print("\nADVERTENCIA METODOLÓGICA (prevención de falacia ecológica):")
print("Que una comuna haya votado mayoritariamente por una lista NO permite afirmar nada")
print("sobre cómo votó una persona de esa comuna. Son datos agregados por puesto de")
print("votación, y un puesto no es un grupo social homogéneo.")


Concejo 2019 · votación por comuna


,cod_comuna,comuna_normalizada,votos,pct
1,01,COMUNA 1,19903,16.3
3,03,COMUNA 3,19255,15.8
5,05,COMUNA 5,18307,15.0
2,02,COMUNA 2,16331,13.4
4,04,COMUNA 4,14689,12.1
7,07,COMUNA 7,10373,8.5
6,06,COMUNA 6,10287,8.4
8,08,CORREGIMIENTO EL CENTRO,8611,7.1
9,09,CORREGIMIENTO EL LLANITO,1833,1.5
10,10,CORREG. LA FORTUNA AUTOPISTA,1144,0.9



Concejo 2023 · votación por comuna


,cod_comuna,comuna_normalizada,votos,pct
3,03,COMUNA 3,20709,15.7
1,01,COMUNA 1,20116,15.2
5,05,COMUNA 5,19688,14.9
2,02,COMUNA 2,16633,12.6
4,04,COMUNA 4,15293,11.6
7,07,COMUNA 7,13000,9.9
6,06,COMUNA 6,11423,8.7
8,08,CORREGIMIENTO EL CENTRO,10367,7.9
9,09,CORREGIMIENTO EL LLANITO,2350,1.8
10,10,CORREG. LA FORTUNA AUTOPISTA,1287,1.0



ADVERTENCIA METODOLÓGICA (prevención de falacia ecológica):
Que una comuna haya votado mayoritariamente por una lista NO permite afirmar nada
sobre cómo votó una persona de esa comuna. Son datos agregados por puesto de
votación, y un puesto no es un grupo social homogéneo.


## 9 · Hallazgos

Cada uno con su nivel de evidencia y con lo que **no** significa, como exige la
metodología del proyecto.

In [12]:
hallazgos = []
if "2023" in resultados:
    r = resultados["2023"]
    u = asg.umbral_real_de_entrada(r, concejo["2023"])
    p = asg.votos_sin_representacion(r)
    i = asg.indicadores_competencia(r)

    hallazgos.append({
        "hallazgo": f"En 2023, {u['no_electos_con_mas_votos_que_el_ultimo_electo']} "
                    f"candidatos al Concejo sacaron más votos que el último electo "
                    f"({miles(u['votos_del_ultimo_electo'])}) y no obtuvieron curul.",
        "nivel_de_evidencia": "Dato descriptivo",
        "que_no_significa": "No significa que hubo un error ni una irregularidad. Es el "
                            "efecto normal de la cifra repartidora: la curul la gana la "
                            "lista, no el candidato.",
    })
    hallazgos.append({
        "hallazgo": f"{miles(p['votos_en_listas_sin_curul'])} votos "
                    f"({p['porcentaje_sobre_validos']} % de los válidos) fueron a "
                    f"{p['listas_sin_curul']} listas que no obtuvieron ninguna curul.",
        "nivel_de_evidencia": "Dato descriptivo",
        "que_no_significa": "No significa que esos votos se hayan perdido por un fallo del "
                            "sistema. Es la consecuencia esperada de un umbral, y el "
                            "umbral existe por diseño del constituyente.",
    })
    hallazgos.append({
        "hallazgo": f"El número efectivo de partidos pasó de "
                    f"{i['numero_efectivo_de_partidos_por_votos']} medido en votos a "
                    f"{i['numero_efectivo_de_partidos_por_curules']} medido en curules.",
        "nivel_de_evidencia": "Patrón estadístico",
        "que_no_significa": "No significa que el sistema favorezca a un partido concreto. "
                            "Es la reducción que produce cualquier fórmula con umbral.",
    })
    hallazgos.append({
        "hallazgo": "Una de las 17 curules fue de lista cerrada y su titular no se puede "
                    "determinar con el archivo de resultados.",
        "nivel_de_evidencia": "Dato descriptivo",
        "que_no_significa": "No significa que el dato esté mal ni que falte información "
                            "oficial. La votación de una lista cerrada es de la lista; el "
                            "orden de los candidatos está en el formulario de inscripción.",
    })

if hallazgos:
    for h in hallazgos:
        print(f"\n{'─' * 66}")
        print(h["hallazgo"])
        print(f"  Nivel de evidencia: {h['nivel_de_evidencia']}")
        print(f"  Qué NO significa:   {h['que_no_significa']}")
    escribir_csv(pd.DataFrame(hallazgos), SALIDA / "hallazgos_concejo.csv")


──────────────────────────────────────────────────────────────────
En 2023, 21 candidatos al Concejo sacaron más votos que el último electo (932) y no obtuvieron curul.
  Nivel de evidencia: Dato descriptivo
  Qué NO significa:   No significa que hubo un error ni una irregularidad. Es el efecto normal de la cifra repartidora: la curul la gana la lista, no el candidato.

──────────────────────────────────────────────────────────────────
32.799 votos (26.23 % de los válidos) fueron a 14 listas que no obtuvieron ninguna curul.
  Nivel de evidencia: Dato descriptivo
  Qué NO significa:   No significa que esos votos se hayan perdido por un fallo del sistema. Es la consecuencia esperada de un umbral, y el umbral existe por diseño del constituyente.

──────────────────────────────────────────────────────────────────
El número efectivo de partidos pasó de 16.0 medido en votos a 9.32 medido en curules.
  Nivel de evidencia: Patrón estadístico
  Qué NO significa:   No significa que el sistema f

## 10 · Estado de la etapa

In [13]:
salidas = sorted(p for p in SALIDA.glob("*.csv"))
estado = {
    "cuaderno": "02_curules",
    "version_cuaderno": VERSION_NB,
    "id_ejecucion": ID_EJECUCION,
    "generado": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "curules_del_concejo": CURULES_CONCEJO,
    "anios_analizados": sorted(resultados),
    "resumen_por_anio": {
        a: {
            "votos_validos": r["votos_validos"],
            "cuociente_electoral": round(r["cuociente_electoral"], 2),
            "umbral": round(r["umbral"], 2),
            "cifra_repartidora": round(r["cifra_repartidora"], 2),
            "listas_con_votos": int(len(r["listas"])),
            "listas_que_pasan_umbral": int(r["listas"]["supera_umbral"].sum()),
            "listas_con_curul": int((r["listas"]["curules"] > 0).sum()),
            "curules_asignadas": r["curules_asignadas"],
            "curules_sin_nombre": r["curules_sin_nombre"],
        } for a, r in sorted(resultados.items())
    },
    "validacion_2023": comp if "2023" in resultados else None,
    "salidas": [rel(p) for p in salidas],
    "pendiente_para_cerrar": [
        "Candidaturas inscritas: resuelve el titular de la curul de lista cerrada y "
        "aporta el género de cada candidatura.",
        "Censo electoral por puesto: sin él no hay participación ni abstención.",
    ],
}
escribir_json(AUDITORIA / "estado_etapa.json", estado)
escribir_json(RAIZ / "docs" / "fuentes" / "estado_curules_ultimo.json", estado)

print(f"Años analizados: {', '.join(sorted(resultados))}")
print(f"Salidas escritas: {len(salidas)} en {rel(SALIDA)}")
for p in salidas:
    print(f"  {p.name}")
print(f"\nAuditoría: {rel(AUDITORIA)}")

Años analizados: 2019, 2023
Salidas escritas: 12 en datos/procesados/concejo
  concentracion_territorial_electos.csv
  electos_concejo_2019.csv
  electos_concejo_2023.csv
  hallazgos_concejo.csv
  indicadores_competencia.csv
  listas_concejo_2019.csv
  listas_concejo_2023.csv
  no_electos_con_mas_votos_2019.csv
  no_electos_con_mas_votos_2023.csv
  votos_por_comuna_concejo_2019.csv
  votos_por_comuna_concejo_2023.csv
  votos_sin_representacion.csv

Auditoría: docs/fuentes/auditorias/02_curules/20261003_235924_506795
